In [ ]:
#1. Load the raw file

import pandas as pd

df = pd.read_csv(
    r"C:\Users\suriya.r\Documents\Network operations and predictive intelligence system\Dataset\archive (1)\sms-call-internet-mi-2013-11-01.csv"
)

df.head()

,datetime,CellID,countrycode,smsin,smsout,callin,callout,internet
0,2013-11-01 00:00:00,1,0,0.3521,NaN,NaN,0.0273,NaN
1,2013-11-01 00:00:00,1,33,NaN,NaN,NaN,NaN,0.0261
2,2013-11-01 00:00:00,1,39,1.7322,1.1047,0.5919,0.4020,57.7729
3,2013-11-01 00:00:00,2,0,0.3581,NaN,NaN,0.0273,NaN
4,2013-11-01 00:00:00,2,33,NaN,NaN,NaN,NaN,0.0274


In [4]:
#2. Canonicalize the column name
column_mapping = {
    "datetime": "timestamp",
    "CellID": "grid_id",
    "countrycode": "country_code",
    "smsin": "sms_in",
    "smsout": "sms_out",
    "callin": "call_in",
    "callout": "call_out",
    "internet": "internet_activity"
}

df1= df.rename(columns=column_mapping)
# print(df.head())

df1.columns

Index(['timestamp', 'grid_id', 'country_code', 'sms_in', 'sms_out', 'call_in',
       'call_out', 'internet_activity'],
      dtype='str')

In [5]:
df1.columns = df1.columns.str.strip().str.lower()

print(df1.columns.tolist())

['timestamp', 'grid_id', 'country_code', 'sms_in', 'sms_out', 'call_in', 'call_out', 'internet_activity']


In [6]:
# 3 Parse timestamp

df1["timestamp"] = pd.to_datetime(
    df1["timestamp"],
    errors="coerce"
)

df1["date"]=df1["timestamp"].dt.date
df1["hour"]=df1["timestamp"].dt.hour
df1["day_of_week"]=df1["timestamp"].dt.dayofweek
print(df1["date"])
print(df1["hour"])
print(df1["day_of_week"].head())

0          2013-11-01
1          2013-11-01
2          2013-11-01
3          2013-11-01
4          2013-11-01
              ...    
1891923    2013-11-01
1891924    2013-11-01
1891925    2013-11-01
1891926    2013-11-01
1891927    2013-11-01
Name: date, Length: 1891928, dtype: object
0           0
1           0
2           0
3           0
4           0
           ..
1891923    23
1891924    23
1891925    23
1891926    23
1891927    23
Name: hour, Length: 1891928, dtype: int32
0    4
1    4
2    4
3    4
4    4
Name: day_of_week, dtype: int32


In [ ]:
# Count distinct timestamps

unique_timestamps = ( df1["timestamp"].dropna()
                     .sort_values() .unique()
)

print( len(unique_timestamps))

Distinct timestamps: 24


In [15]:
timestamp_series = pd.Series(unique_timestamps)

differences = timestamp_series.diff().dropna()

print(differences.value_counts())

0 days 01:00:00    23
Name: count, dtype: int64


In [ ]:
hourly_cadence_valid = (
    len(unique_timestamps) == 24
    and (differences == pd.Timedelta(hours=1)).all())

print("Hourly cadence valid:", hourly_cadence_valid)

Hourly cadence valid: True


In [ ]:
# 4. Check missing values, blanks, duplicates and negatives 

missing_grid_id = df1["grid_id"].isna().sum()

print("Missing grid_id:", missing_grid_id)

Missing grid_id: 0


In [19]:
missing_timestamp = df1["timestamp"].isna().sum()

print("Missing timestamp:", missing_timestamp)

Missing timestamp: 0


In [ ]:
activity_cols = [
    "sms_in",
    "sms_out",
    "call_in",
    "call_out",
    "internet_activity"
]

for col in activity_cols:
    blank_count = (df1[col].astype(str)
        .str.strip().eq("").sum())

    print(f"{col}: {blank_count} blank values")


print(
    df1[activity_cols]
    .isna()
    .sum()
)

sms_in: 0 blank values
sms_out: 0 blank values
call_in: 0 blank values
call_out: 0 blank values
internet_activity: 0 blank values
sms_in               1086153
sms_out              1422446
call_in              1407781
call_out             1037413
internet_activity    1087074
dtype: int64


In [25]:
exact_duplicates = df1.duplicated().sum()

print("Exact duplicate rows:", exact_duplicates)

Exact duplicate rows: 0


In [ ]:
negative_counts = (df1[activity_cols] < 0).sum()

print(negative_counts)

sms_in               0
sms_out              0
call_in              0
call_out             0
internet_activity    0
dtype: int64


In [27]:
# 5. Inspect multiple country-code rows for the same grid/hour

grid_hour_counts = (
    df1
    .groupby(["timestamp", "grid_id"])
    .size()
    .reset_index(name="row_count")
)

multiple_country_rows = grid_hour_counts[
    grid_hour_counts["row_count"] > 1
]

print(multiple_country_rows.head())


   timestamp  grid_id  row_count
0 2013-11-01        1          3
1 2013-11-01        2          3
2 2013-11-01        3          3
3 2013-11-01        4          3
4 2013-11-01        5          3


In [28]:
example = multiple_country_rows.iloc[0]

example_timestamp = example["timestamp"]
example_grid = example["grid_id"]

print(
    df1[
        (df1["timestamp"] == example_timestamp) &
        (df1["grid_id"] == example_grid)
    ][
        ["timestamp", "grid_id", "country_code"]
    ]
)

   timestamp  grid_id  country_code
0 2013-11-01        1             0
1 2013-11-01        1            33
2 2013-11-01        1            39


In [31]:
# 6 Create total_sms, total_calls and total_activity

df1["total_sms"] = (df1["sms_in"] + df1["sms_out"] )
df1["total_calls"] = (df1["call_in"] + df1["call_out"] )
df1["total_activity"] = (
    df1["total_sms"]
    + df1["total_calls"]
    + df1["internet_activity"]
)
print(df1["total_sms"])


0             NaN
1             NaN
2          2.8369
3             NaN
4             NaN
            ...  
1891923       NaN
1891924       NaN
1891925    6.9955
1891926       NaN
1891927       NaN
Name: total_sms, Length: 1891928, dtype: float64


In [ ]:
# 7.Compute the profiling facts
# A. Number of unique grids

unique_grids=df1["grid_id"].unique()

print("unique grids",unique_grids)

unique grids [    1     2     3 ...  9998  9999 10000]


In [ ]:
# Time range


start_time = df1["timestamp"].min()

end_time = df1["timestamp"].max()

print("Time range:", start_time, "to", end_time)

Time range: 2013-11-01 00:00:00 to 2013-11-01 23:00:00


In [ ]:
# Cadence
# D. Country-code categories
country_categories = (
    df1["country_code"]
    .dropna()
    .unique()
)

print("Country-code categories:", country_categories)

Country-code categories: [    0    33    39    64    46    32    40    34    49    81     7   593
   972    20    44    41     1    86    31    53    63   880    92   212
   221    43   351    48    51   380    55    58    62    45   503   353
    30   233    47    57   242    90   421   420   973   385  1780    84
   373   372    52   216    54    36    82   213    98 88239   595    61
   370   356   354    93   591   856   258   352   852  1514    94   357
   358  1787   359  1829    60   598   381   244  1671   974   371   389
   971   225   234   966   355   382    65   507   375    56    66   238
  1519   218  1613  8817   226 18098   241   386 18099   223   994 18093
    91   965  1416  1579  1778  1705   886   237   976    95   675   230
  1849  1647   265   504   243   968  7705  1438  1587 18096  7778   261
   255  1214   506    27   251 18094   263  1905  7702  7707  7701   257
   228   590 18762   962   998   291   254  1604  1403   992  7775   961
   377   374   853   995  

In [ ]:
print(df1["country_code"].value_counts())

country_code
39      240000
0       231307
46      109848
33      106603
49       95933
         ...  
1938         4
247          3
240          2
596          2
1709         2
Name: count, Length: 246, dtype: int64


In [ ]:
# Busiest hourly window

hourly_activity = (
    df1.groupby("timestamp")["total_activity"]
    .sum()
    .sort_values(ascending=False)
)

In [37]:
busiest_hour = hourly_activity.idxmax()
busiest_activity = hourly_activity.max()

print("Busiest hour:", busiest_hour)
print("Activity:", busiest_activity)

Busiest hour: 2013-11-01 11:00:00
Activity: 5009588.4424


In [ ]:
# Busiest grid

grid_activity = (
    df1
    .groupby("grid_id")["total_activity"]
    .sum()
    .sort_values(ascending=False)
)

busiest_grid = grid_activity.idxmax()
busiest_grid_activity = grid_activity.max()

print("Busiest grid:", busiest_grid)
print("Activity:", busiest_grid_activity)

Busiest grid: 5161
Activity: 262060.34


In [ ]:
# Null counts per column

null_counts=df1.isnull().sum()

print(null_counts)

timestamp                  0
grid_id                    0
country_code               0
sms_in               1086153
sms_out              1422446
call_in              1407781
call_out             1037413
internet_activity    1087074
date                       0
hour                       0
day_of_week                0
total_sms            1502892
total_calls          1594716
total_activity       1658462
dtype: int64
